# autoresearch — inference experiment analysis

Load `results.tsv` and visualize progress across experiments.

**Primary metric:** `score` (correct GSM8K problems solved within 5-min budget) — higher is better.  
**Secondary metrics:** `accuracy`, `tokens_per_sec`

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


plt.rcParams.update(
    {
        "figure.facecolor": "#0d1117",
        "axes.facecolor": "#0d1117",
        "axes.edgecolor": "#30363d",
        "axes.labelcolor": "#c9d1d9",
        "xtick.color": "#8b949e",
        "ytick.color": "#8b949e",
        "text.color": "#c9d1d9",
        "grid.color": "#21262d",
        "grid.linewidth": 0.8,
        "font.size": 11,
    }
)

df = pd.read_csv("results.tsv", sep="\t")
df["exp_num"] = range(1, len(df) + 1)
print(f"{len(df)} experiments loaded")
df.head(10)

In [ ]:
# Experiment outcome counts
counts = df["status"].value_counts()
colors = {"keep": "#238636", "discard": "#da3633", "crash": "#9e6a03"}

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(
    counts.index, counts.values, color=[colors.get(s, "#58a6ff") for s in counts.index], edgecolor="none", width=0.5
)
ax.bar_label(bars, fontsize=12, color="#c9d1d9", padding=4)
ax.set_title("Experiment outcomes", fontsize=13, color="#e6edf3", pad=10)
ax.set_ylabel("count")
ax.set_ylim(0, counts.max() * 1.25)
ax.grid(axis="y", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Score progression over experiments
kept = df[df["status"] == "keep"].copy()
kept["running_best"] = kept["score"].cummax()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: score over time with frontier
ax = axes[0]
all_kept = df[df["status"] == "keep"]
all_disc = df[df["status"] == "discard"]

ax.scatter(all_disc["exp_num"], all_disc["score"], color="#da3633", alpha=0.5, s=30, label="discard", zorder=3)
ax.scatter(all_kept["exp_num"], all_kept["score"], color="#238636", s=50, label="keep", zorder=4)
ax.step(
    kept["exp_num"], kept["running_best"], where="post", color="#58a6ff", linewidth=2, label="best so far", zorder=5
)

# Annotate top improvements
improved = kept[kept["score"] == kept["running_best"]]
for _, row in improved.iterrows():
    ax.annotate(
        row["description"][:28],
        xy=(row["exp_num"], row["score"]),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=7,
        color="#8b949e",
        ha="left",
    )

ax.set_title("Score progression", fontsize=13, color="#e6edf3", pad=10)
ax.set_xlabel("experiment #")
ax.set_ylabel("score (correct problems)")
ax.legend(fontsize=9, framealpha=0.2)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Right: accuracy vs tokens_per_sec scatter
ax2 = axes[1]
sc = ax2.scatter(
    all_kept["tokens_per_sec"], all_kept["accuracy"], c=all_kept["score"], cmap="YlGn", s=60, zorder=4, label="keep"
)
ax2.scatter(
    all_disc["tokens_per_sec"], all_disc["accuracy"], color="#da3633", alpha=0.4, s=30, zorder=3, label="discard"
)
plt.colorbar(sc, ax=ax2, label="score")
ax2.set_title("Accuracy vs throughput", fontsize=13, color="#e6edf3", pad=10)
ax2.set_xlabel("tokens / sec")
ax2.set_ylabel("accuracy")
ax2.legend(fontsize=9, framealpha=0.2)
ax2.grid(True, alpha=0.3)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight", facecolor="#0d1117", edgecolor="none")
plt.show()

In [ ]:
# Summary statistics
kept = df[df["status"] == "keep"]
baseline = kept.iloc[0] if len(kept) > 0 else None
best = kept.loc[kept["score"].idxmax()] if len(kept) > 0 else None

print("=== Summary ===")
print(f"Total experiments:   {len(df)}")
print(f"  kept:              {len(df[df['status'] == 'keep'])}")
print(f"  discarded:         {len(df[df['status'] == 'discard'])}")
print(f"  crashed:           {len(df[df['status'] == 'crash'])}")
print()
if baseline is not None:
    print(
        f"Baseline score:      {baseline['score']}  (acc: {baseline['accuracy']:.4f}, tok/s: {baseline['tokens_per_sec']:.1f})"
    )
if best is not None:
    delta = best["score"] - baseline["score"] if baseline is not None else 0
    pct = 100 * delta / baseline["score"] if baseline is not None and baseline["score"] > 0 else 0
    print(f"Best score:          {best['score']}  ({delta:+d}, {pct:+.1f}%)")
    print(f"Best experiment:     {best['description']}")
    print(f"Best commit:         {best['commit']}")

In [ ]:
# Top hits ranked by score improvement over baseline
kept = df[df["status"] == "keep"].copy()
if len(kept) > 1:
    baseline_score = kept.iloc[0]["score"]
    kept["delta"] = kept["score"] - baseline_score
    top = kept.sort_values("delta", ascending=False).head(10)
    print("Top experiments by score improvement over baseline:")
    print(top[["commit", "score", "delta", "accuracy", "tokens_per_sec", "description"]].to_string(index=False))
else:
    print("Need at least 2 kept experiments to rank improvements.")